                                                __python implementation__

In [1]:
import pickle   # lets you save a trained Python object (like a model) to a file, and load it back later. Analogy: freezing cooked food so you can reheat it later instead of cooking from scratch.
import pandas as pd  # importing pandas for analyzing, cleaning, exploring, and manipulating data
from sklearn.model_selection import train_test_split  # importing train_test_split for spliting the data
from preprocess1 import *  # this pulls in everything from your preprocess1.py file (the divide_by_12 function, ModifiedLabelEncoder class, etc.). The * means "import everything," which works but is considered slightly risky in real projects because you can't see exactly what names you're bringing in.
from imblearn.over_sampling import SMOTE  # a technique to fix imbalanced data (explained below).

In [2]:
df=pd.read_csv("loan_approved.csv")  # Loading dataset

In [3]:
df.columns  # Checking coloums of datasets

Index(['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education',
       'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History', 'Property_Area',
       'Loan_Status (Approved)'],
      dtype='object')

In [4]:
df

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status (Approved)
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...
609,LP002978,Female,No,0,Graduate,No,2900,0.0,71.0,360.0,1.0,Rural,Y
610,LP002979,Male,Yes,3+,Graduate,No,4106,0.0,40.0,180.0,1.0,Rural,Y
611,LP002983,Male,Yes,1,Graduate,No,8072,240.0,253.0,360.0,1.0,Urban,Y
612,LP002984,Male,Yes,2,Graduate,No,7583,0.0,187.0,360.0,1.0,Urban,Y


In [5]:
for column in df.columns:
    print(f"\n--- {column} ---")
    print(df[column].value_counts())


--- Loan_ID ---
Loan_ID
LP001002    1
LP002328    1
LP002305    1
LP002308    1
LP002314    1
           ..
LP001692    1
LP001693    1
LP001698    1
LP001699    1
LP002990    1
Name: count, Length: 614, dtype: int64

--- Gender ---
Gender
Male      489
Female    112
Name: count, dtype: int64

--- Married ---
Married
Yes    398
No     213
Name: count, dtype: int64

--- Dependents ---
Dependents
0     345
1     102
2     101
3+     51
Name: count, dtype: int64

--- Education ---
Education
Graduate        480
Not Graduate    134
Name: count, dtype: int64

--- Self_Employed ---
Self_Employed
No     500
Yes     82
Name: count, dtype: int64

--- ApplicantIncome ---
ApplicantIncome
2500    9
4583    6
6000    6
2600    6
3333    5
       ..
3244    1
4408    1
3917    1
3992    1
7583    1
Name: count, Length: 505, dtype: int64

--- CoapplicantIncome ---
CoapplicantIncome
0.0       273
2500.0      5
2083.0      5
1666.0      5
2250.0      3
         ... 
2791.0      1
1010.0      1
1695.0  

In [6]:
df.isnull().sum()  # Checking null values

Loan_ID                    0
Gender                    13
Married                    3
Dependents                15
Education                  0
Self_Employed             32
ApplicantIncome            0
CoapplicantIncome          0
LoanAmount                22
Loan_Amount_Term          14
Credit_History            50
Property_Area              0
Loan_Status (Approved)     0
dtype: int64

# Imputing Null values - Filling in missing values

replacing missing values with a reasonable substitute instead of deleting rows.

Why: Machine learning models cannot handle blank/NaN cells — the math simply breaks. You have two choices: drop the rows (lose data) or fill them in (keep data, accept some guessing). Here, every gap is filled instead of dropped, so no rows are lost.

df.loc[condition, 'column'] = value reads as: "find rows where condition is True, and in those rows, set column to value."

Categorical columns (Gender, Married, Self_Employed) are filled with the most frequent value seen in that column — this is called mode imputation. Analogy: if you don't know someone's favorite color and 80% of your friends said blue, guessing "blue" is your safest bet.

LoanAmount uses the median (the middle value when sorted) rather than the mean (average), because loan amounts often have outliers (a few huge loans), and the median resists being dragged by extreme values.

Loan_Amount_Term and Credit_History are filled with fixed constants (360.0 and 0.0) — these look like they were chosen after inspecting the data separately (360.0 is probably the most common loan term; 0.0 for Credit_History likely means "assume no credit history" as the safe default).

In [7]:
df.loc[df['Gender'].isnull(),'Gender']="Male"  # Filling null value using Most_Frequent Elements of Gender column.

In [8]:
df.loc[df['Married'].isnull(),'Married']="Yes"    # Filling null value using Most_Frequent Elements of Married column.

In [9]:
df.loc[df['Dependents'].isnull(),'Dependents']='0'   # Filling null value using Most_Frequent Elements of Dependents

In [10]:
df.loc[df['Self_Employed'].isnull(),'Self_Employed']='No'   # Filling null value using Most_Frequent Elements of Self_Employed

In [11]:
#  Filling null values using Median of LoanAmount
df.loc[df['LoanAmount'].isnull(),'LoanAmount']=df['LoanAmount'].median()

In [12]:
#  Filling null values using Most_Frequent value of Loan_Amount_Term column
df.loc[df['Loan_Amount_Term'].isnull(),'Loan_Amount_Term']=360.0

In [13]:
df.loc[df['Credit_History'].isnull(),'Credit_History']=0.0    # Filling null values of Credit_History

In [14]:
df.isnull().sum()   # Checking null values are available or not

Loan_ID                   0
Gender                    0
Married                   0
Dependents                0
Education                 0
Self_Employed             0
ApplicantIncome           0
CoapplicantIncome         0
LoanAmount                0
Loan_Amount_Term          0
Credit_History            0
Property_Area             0
Loan_Status (Approved)    0
dtype: int64

# Loading preprocessor

preprocessor is probably a preprocessing object or pipeline that was created earlier.

What: Loading a previously-built preprocessing pipeline from disk instead of rebuilding it here.

Why: In real projects, preprocessing (scaling numbers, encoding categories) is built once — often in a separate notebook (that's your preprocessing.ipynb) — and saved so it can be reused consistently on both training and future data. This guarantees the exact same transformations are applied every time, rather than risking small inconsistencies if you rewrote the logic here.

Code breakdown: "rb" means "read binary" — pickle files aren't plain text, so you must open them in binary mode. pickle.load(f) reconstructs the Python object (here, a scikit-learn pipeline/ColumnTransformer) exactly as it was when saved.

Output : Printing preprocessor shows the internal structure of the pipeline — which columns get scaled, which get encoded, etc.
_________________________________________________________________________________________________________________________

A .pkl file is a Pickle file in Python.

Pickle allows Python objects to be saved to a file so they can be loaded and reused later.
```
For example, you might have created a preprocessing pipeline like:

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_columns),
    ("cat", OneHotEncoder(), categorical_columns)
])

After creating and fitting it, you could save it:

with open("preprocessing1.pkl", "wb") as f:
    pickle.dump(preprocessor, f)

Now all those preprocessing settings are stored inside:

preprocessing1.pkl

In [19]:
# Load a preprocessor object from a pickled file
with open("preprocessing1.pkl","rb") as f:
    preprocessor=pickle.load(f)

In [20]:
preprocessor  # Preprocessor or PipeLine

,transformers,"[('OHE columns', ...), ('Label_encoder', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,None
,sparse_output,True


# Splitting the data

In [21]:
# Spliting the data into train and test
train,test,_,_=train_test_split(df,df['LoanAmount'],test_size=0.2)

Code breakdown: train_test_split normally returns 4 things when given features and a target separately (X_train, X_test, y_train, y_test). Here, the "target" passed in is df['LoanAmount'], but notice the two return values for that target are thrown away with _, _ — the only things actually kept are train and test, which are full-DataFrame splits. This is a bit unusual — passing df['LoanAmount'] as if it's the target when it's actually about to be split off later, inside the preprocessor. It works, but it's a slightly confusing way to just split df itself.

Output (Cell 19): train['Loan_Status (Approved)'].value_counts() shows how many approved vs. rejected loans exist in the training set — this is where you'd notice a class imbalance (e.g., far more "approved" than "rejected"), which sets up the next phase.

Exam point: test_size=0.2 is the most common convention (80/20 split). Also — no random_state is set here, meaning every time this notebook is re-run, you'll get a different random split. That's worth remembering: for reproducible results, you'd normally add random_state=42 (or any fixed number).

In [22]:
train['Loan_Status (Approved)'].value_counts()  # Checking count of unique values of Loan_Status (Approved) columns

Loan_Status (Approved)
Y    338
N    153
Name: count, dtype: int64

# Transforming the data

In [23]:
# Transform the training data using the preprocessor object or PipeLine
processed_data=preprocessor.fit_transform(train)

In [24]:
processed_data

array([[1., 0., 0., ..., 1., 1., 1.],
       [1., 0., 0., ..., 0., 1., 1.],
       [1., 0., 0., ..., 0., 0., 0.],
       ...,
       [1., 0., 0., ..., 3., 1., 0.],
       [1., 0., 0., ..., 2., 1., 1.],
       [1., 0., 0., ..., 0., 1., 1.]], shape=(491, 16))

In [25]:
# Extract the features (all columns except the last one) from the processed data
x_train=processed_data[:,:-1]

In [26]:
# Extract the target variable (last column) from the processed data
y_train=processed_data[:,-1] # Last column is target

What: Apply the preprocessing pipeline to the training data, then split the result into features (x_train) and target (y_train).

Why: Raw columns like "Male"/"Female" or "Yes"/"No" mean nothing to a mathematical model — everything must become numbers first (encoding categories, scaling numeric ranges). fit_transform does two jobs at once: fit learns things like "what are the categories" or "what's the mean/std to scale by," then transform applies that.

Code breakdown: processed_data[:, :-1] uses NumPy-style slicing: "all rows, all columns except the last one." processed_data[:, -1] means "all rows, only the last column." This only works because the preprocessing pipeline was clearly built to put the target column last, after transforming everything.

Output: A NumPy array of numbers — no column names anymore, just raw numeric data ready for a model.

Exam point: Critical rule to remember — you fit_transform on training data (Cell 21) but only transform (no fit) on test data (Cell 39, later). If you fit on test data too, you leak information from the test set into your preprocessing, which is called data leakage, and it's one of the most commonly asked interview questions in ML.

In [27]:
y_train

array([1., 1., 0., 1., 1., 1., 0., 1., 1., 1., 1., 1., 1., 1., 0., 0., 1.,
       1., 0., 1., 1., 1., 0., 1., 1., 0., 1., 1., 1., 1., 0., 0., 1., 1.,
       1., 1., 1., 0., 0., 1., 1., 1., 0., 0., 1., 1., 1., 0., 1., 0., 1.,
       0., 1., 1., 1., 1., 1., 0., 1., 1., 0., 0., 1., 0., 0., 0., 1., 1.,
       0., 1., 1., 0., 1., 0., 1., 0., 0., 1., 1., 1., 0., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 0., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 0.,
       1., 1., 1., 1., 1., 1., 1., 0., 1., 0., 0., 0., 1., 1., 1., 1., 1.,
       0., 1., 1., 1., 1., 1., 0., 1., 0., 1., 1., 0., 1., 0., 1., 1., 1.,
       1., 0., 0., 0., 0., 0., 1., 1., 1., 1., 1., 0., 1., 1., 1., 1., 1.,
       0., 1., 0., 1., 1., 1., 0., 1., 1., 1., 1., 0., 1., 1., 0., 0., 0.,
       1., 1., 1., 0., 0., 0., 1., 0., 1., 0., 1., 1., 1., 1., 0., 1., 0.,
       1., 1., 1., 1., 1., 0., 0., 1., 1., 0., 1., 0., 1., 0., 1., 1., 0.,
       0., 0., 1., 1., 0., 1., 1., 0., 1., 1., 1., 1., 1., 1., 1., 0., 1.,
       1., 1., 1., 1., 0.

# Balancing the data

## SMOTE

In [28]:
#!pip install imblearn

In [29]:
# Initialize the SMOTE (Synthetic Minority Over-sampling Technique) object into variable
smote=SMOTE()

In [30]:
# Apply SMOTE to the training data to balance the class distribution
x_train_smote,y_train_smote=smote.fit_resample(x_train,y_train)

In [31]:
len(y_train)  # Checking length of actual y_train

491

In [32]:
len(y_train_smote)   # Checking length of y_train after apply Smote

676

Exam point: SMOTE is applied only to the training data, never to test data — the test set must reflect the real world's actual imbalance, or your evaluation becomes meaningless. This is a frequently asked interview trap question.

# Model Building

SVM is the Supervised Machine Learning algorithm used for both classification, regression. But mostly preferred for classification.

Given a dataset, the algorithm tries to divide the data using hyperplanes and then makes the predictions. SVM is a non-probabilistic linear classifier. While other classifiers, when classifying, predict the probability of a data point to belong to one group or the another, SVM directly says to which group the datapoint belongs to without using any probability calculation.

How it works?

SVM constructs a best line or the decision boundary called Hyperplane which can be used for classification or regression or outlier detection. The dimension of the hyperplane depends upon the number of features. If the number of input features is 2, then the hyperplane is just a line. If the number of input features is 3, then the hyperplane becomes a two-dimensional plane.

This hyperplane creates 2 margin lines parallel to it which have some distance so that it can distinctly classify the data points. The distance between the 2 margin lines are called marginal distance.

These 2 margin lines passes through the most nearest +ve points and the most nearest -ve points. Those points through which the margin lines pass are called support vectors. Support vectors are important as it helps to determine the maximum distance of the marginal plane.

## Understanding the Mathematics involved

Let’s take the example of the following dataset and see how can we divide the data into appropriate groups.

We can see that there are two groups of data. The question is how to divide these points into two groups. It can be done using any of the three lines. Or, for that purpose, there can be an infinite number of straight lines that can divide these points into two classes. Now, which line to choose? SVM solves this problem using the maximum margin as shown

The black line in the middle is the optimum classifier. This line is drawn to maximise the distance of the classifier line from the nearest points in the two classes. It is also called a hyperplane in terms of SVM. A Hyperplane is an n-1 dimensional plane which optimally divides the data of n dimensions. Here, as we have only a 2-D data, so the hyperplane can be represented using one dimension only. Hence, the hyperplane is a line here. The two points (highlighted with circles) which are on the yellow lines, they are called the support vectors. As it is a 2-D figure, they are points. In a multi-dimensional space, they will be vectors, and hence, the name- support vector machine as the algorithm creates the optimum classification line by maximising its distance from the two support vectors.

When the data is not linearly separable, then to create a hyperplane to separate data into different groups, the SVM algorithm needs to perform computations in a higher-dimensional space. But the introduction of new dimensions makes the computations for the SVMs more intensive, which impacts the algorithm performance. To rectify this, mathematicians came up with the approach of Kernel methods. Kernel methods use kernel functions available in mathematics. The unique feature of a kernel function is to compute in a higher-dimensional space without calculating the new coordinates in that higher dimension. It implicitly uses predefined mathematical functions to do operations on the existing points which mimic the computation in a higher-dimensional space without adding to the computation cost as they are not actually calculating the coordinates in the higher dimension thereby avoiding the computation of calculating distances from the newly computed points. This is called the kernel trick.  Image: bogotobogo.com

In the left diagram above, we have a non-linear distribution of data as we can not classify a data using a linear equation. To solve this problem, we can project the points in a 3-dimensional space and then derive a plane which divides the data into two parts. In theory, that’s what a kernel function does without computing the additional coordinates for the higher dimension.

In [33]:
from sklearn.svm import SVC  # # assign Support vector classifier
svclassifier = SVC() ## base model with default parameters
svclassifier.fit(x_train_smote,y_train_smote)  ## Fit the SVC to the resampled training data

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


SVM tries to draw the best possible dividing line (called a hyperplane) between two classes — imagine drawing a line on a piece of paper to separate red dots from blue dots. It doesn't just draw any separating line — it picks the one that keeps the largest possible gap (margin) from the nearest points of each class. Those nearest points are called support vectors — they're the only points that actually matter for positioning the line; every other point could be deleted and the line wouldn't move.

Code breakdown: .fit(x_train_smote, y_train_smote) is where the actual learning happens — the model looks at the balanced training data and works out where to place the hyperplane.

Output: A summary line describing the fitted SVC and its default settings (kernel='rbf', C=1.0, etc.).

Exam point: Be ready to explain, in plain words: "SVM finds the boundary that maximizes the margin between classes, using only the closest points (support vectors)." Also know that when data isn't linearly separable, SVM uses kernels (like RBF) to handle curved boundaries without literally computing extra dimensions — that's the "kernel trick," and it's a guaranteed interview question.

# Validating the model

### Proessing the test data

In [34]:
test_processed=preprocessor.transform(test)  # Transform the test data using the preprocessor # # transform only, no fit!

In [35]:
x_test=test_processed[:,:-1]  # Extract the features (all columns except the last one) from the processed data
y_test=test_processed[:,-1]   # Extract the target variable (last column) from the processed data

In [36]:
y_test

array([1., 0., 1., 1., 1., 1., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1., 1., 1., 1., 1., 1., 0.,
       0., 1., 0., 1., 0., 1., 1., 1., 0., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 0., 1., 0., 1., 1., 1., 1., 1., 0., 1., 0., 1., 0., 1., 1., 1.,
       0., 0., 0., 1., 0., 1., 0., 0., 1., 0., 0., 0., 1., 0., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 0., 0., 1., 1., 0., 1., 1., 0., 0., 1., 1.,
       1., 1., 1., 0., 0., 1., 1., 0., 0., 1., 0., 1., 1., 1., 1., 0., 1.,
       0., 0., 1., 0.])

In [37]:
# Getting predictions from model
y_pred=svclassifier.predict(x_test)

### Evaluating the model performance

In [38]:
# Importing the classification_report function from sklearn.metrics
from sklearn.metrics import classification_report
# Printing the classification report comparing the true labels (y_test) and the predicted labels (y_pred)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

         0.0       0.47      0.23      0.31        39
         1.0       0.71      0.88      0.79        84

    accuracy                           0.67       123
   macro avg       0.59      0.56      0.55       123
weighted avg       0.64      0.67      0.64       123



# Hyper Parameter Tuning

## What is a Model Hyperparameter?

A model hyperparameter is a configuration that is external to the model and whose value cannot be estimated from data.



## Hyperparameters of Support Vector Machine

#### SVM separates data points that belong to different classes with a decision boundary. When determining the decision boundary, a soft margin SVM (soft margin means allowing some data points to be misclassified) tries to solve an optimization problem with the following goals:

#### 1)Increase the distance of decision boundary to classes (or support vectors)
#### 2)Maximize the number of points that are correctly classified in the training set

### There is obviously a trade-off between these two goals which and it is controlled by C which adds a penalty for each misclassified data point.

### If C is small, the penalty for misclassified points is low so a decision boundary with a large margin is chosen at the expense of a greater number of misclassification.

### If C is large, SVM tries to minimize the number of misclassified examples due to the high penalty which results in a decision boundary with a smaller margin. The penalty is not the same for all misclassified examples. It is directly proportional to the distance to the decision boundary.

### Gamma is a hyperparameter used with non-linear SVM. One of the most commonly used non-linear kernels is the radial basis function (RBF). Gamma parameter of RBF controls the distance of the influence of a single training point.

### Low values of gamma indicate a large similarity radius which results in more points being grouped together.

### For high values of gamma, the points need to be very close to each other in order to be considered in the same group (or class). Therefore, models with very large gamma values tend to overfit.

## GridSearchCV

#### It is the process of performing hyperparameter tuning in order to determine the optimal values for a given model. As mentioned above, the performance of a model significantly depends on the value of hyperparameters

#### Doing this manually could take a considerable amount of time and resources and thus we use GridSearchCV to automate the tuning of hyperparameters.

GridSearchCV tries all the combinations of the values passed in the m

dictionary and evaluates the model for each combination using the Cross-Validation method. Hence after using this function we get accuracy/loss for every combination of hyperparameters and we can choose the one with the best performance.

- gamma = 'auto' ![image.png](attachment:924fc53c-a5eb-4078-ab44-4a713bb08a45.png)
- gamma = 'scale' ![image.png](attachment:45dd628f-ae44-4b59-a610-311f3e18b7b6.png)

In [39]:
from itertools import product  # Importing the product function from itertools module

param_grid_linear = {           #  Define Parameter grid for linear kernel SVM
    'C': [0.1, 5, 10,50,60,70],
    'kernel': ['linear'],
    'gamma': ['scale', 'auto']
}
param_grid_rbf = {              # Define Parameter grid for Radial Basic Function-RBF
    'C': [0.1, 5, 10,50,60,70],
    'kernel': ['rbf'],
    'gamma': ['scale', 'auto']
}
param_grid_poly = {             # Define Parameters grid for Polynomial kernel
    'C': [0.1, 5, 10,50,60,70],
    'kernel': ['poly'],
    'gamma': ['scale', 'auto'],
    'degree': [2, 3, 4]
}

In [40]:
# Importing GridSearchCV from sklearn
from sklearn.model_selection import GridSearchCV

# Assigning SVC model into variables
model=SVC()

# Defining the grid search using GridSearchCV
# - The parameter grid is defined by param_grid_poly
# - 'refit=True' ensures that the best estimator found during the grid search is refitted on the whole dataset
# - 'verbose=2' controls the verbosity of the grid search process (higher values result in more output)
# - 'scoring='f1'' specifies the scoring metric for evaluating the model's performance during grid search
# - 'cv=5' specifies 5-fold cross-validation for evaluating each combination of hyperparameters
grid = GridSearchCV(model,param_grid=param_grid_rbf, refit = True, verbose = 1,scoring='f1',cv=3)

# fitting the model for grid search
grid.fit(x_train_smote,y_train_smote)

Fitting 3 folds for each of 12 candidates, totalling 36 fits


,estimator,SVC()
,param_grid,"{'C': [0.1, 5, ...], 'gamma': ['scale', 'auto'], 'kernel': ['rbf']}"
,scoring,'f1'
,n_jobs,None
,refit,True
,cv,3
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,C,70


In [41]:
# Opening a file named "model.pkl" in write-binary mode
# The 'wb' mode is used for writing binary data to the file
with open("model.pkl","wb") as f:
    pickle.dump(grid,f)      # Using pickle to serialize and save the grid search object to the file

In [42]:
print(grid.best_params_)   # Printing the best parameters found by the grid search

{'C': 70, 'gamma': 'auto', 'kernel': 'rbf'}


In [43]:
grid

,estimator,SVC()
,param_grid,"{'C': [0.1, 5, ...], 'gamma': ['scale', 'auto'], 'kernel': ['rbf']}"
,scoring,'f1'
,n_jobs,None
,refit,True
,cv,3
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,C,70


In [47]:
svclassifier = SVC(C=10, gamma='auto',kernel='rbf' ) ## base model with default parameters
svclassifier.fit(x_train_smote,y_train_smote)

,C,10
,kernel,'rbf'
,degree,3
,gamma,'auto'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [48]:
y_hat=grid.predict(x_test)  # Use the best model from grid search to make predictions on the test set

In [49]:
# Generate a classification report comparing true labels (y_test) with predicted labels (y_hat)
print(classification_report(y_test,y_hat))

              precision    recall  f1-score   support

         0.0       0.51      0.54      0.53        39
         1.0       0.78      0.76      0.77        84

    accuracy                           0.69       123
   macro avg       0.65      0.65      0.65       123
weighted avg       0.70      0.69      0.69       123

